# 00 · Guía de uso de los cuadernos (Jupyter)

**Objetivos de aprendizaje**

- Reconocer la interfaz de JupyterLab y los tipos de celda.
- Ejecutar, editar y reordenar celdas con atajos de teclado.
- Comprender el estado del *kernel* y la importancia del orden de ejecución.
- Interpretar mensajes de error y recuperarse de ellos.
- Cargar un dataset propio en la carpeta del taller.
- Comprobar que Spark funciona desde un cuaderno.

| Aspecto | Detalle |
|---|---|
| Duración estimada | 30 minutos |
| Requisito previo | Instalación completada (guías `docs/01` a `docs/05`) |
| Entrada | Ninguna |
| Salida | Entorno verificado |

## 1. Qué es un cuaderno de Jupyter

Un cuaderno de Jupyter (*notebook*) es un documento que combina texto explicativo, código ejecutable y los resultados de ese código (tablas, gráficos, mensajes) en un mismo archivo con extensión `.ipynb`. Kluyver et al. (2016) lo describen como un formato de publicación para flujos de trabajo computacionales reproducibles: quien recibe el cuaderno puede leer el razonamiento y volver a ejecutar cada paso.

En este taller, cada cuaderno es una guía práctica. El texto explica el concepto de Spark que se trabaja, el código lo aplica sobre el dataset activo y los resultados permiten interpretar lo ocurrido.

### Componentes principales

| Componente | Función |
|---|---|
| **Celda de código** | Contiene instrucciones de Python. Al ejecutarse, su resultado aparece debajo. |
| **Celda Markdown** | Contiene texto con formato (títulos, listas, tablas, enlaces). Es la que se está leyendo ahora. |
| **Kernel** | Proceso de Python que ejecuta el código y conserva las variables en memoria. |
| **Contador `[n]`** | Número que aparece a la izquierda de cada celda ejecutada; indica el orden en que se ejecutó. |
| **Salida** | Resultado de la celda: texto, tabla, gráfico o mensaje de error. |

### Interfaz de JupyterLab

```
+-------------------------------------------------------------------+
| Menú: File  Edit  View  Run  Kernel  Tabs  Settings  Help          |
+------------+------------------------------------------------------+
| Explorador | [+] [Guardar] [Cortar] [Copiar] [Pegar] [>] [■] [↻]  |  <- barra del cuaderno
| de archivos|------------------------------------------------------|
|            | [1]: celda de código                                 |
| notebooks/ |      salida                                          |
| data/      | celda Markdown (texto)                               |
| scripts/   | [ ]: celda pendiente de ejecutar                     |
|            |                                       Kernel: Python 3 |
+------------+------------------------------------------------------+
```

- El **explorador de archivos** (izquierda) permite abrir cuadernos y **cargar archivos** con el botón de flecha hacia arriba (*Upload*).
- La **barra del cuaderno** contiene los botones para agregar una celda (+), ejecutar (>), detener (■) y reiniciar el kernel (↻).
- La esquina superior derecha indica el estado del kernel: un círculo vacío significa inactivo; uno relleno, que está ejecutando código (Project Jupyter, s.f.).

## 2. Ejecución de celdas

Para ejecutar una celda se hace clic sobre ella y se presiona `Shift + Enter`. La celda se ejecuta y el cursor pasa a la siguiente. La celda siguiente es un primer ejercicio.

In [ ]:
print("Hola, Spark. El kernel de Python está funcionando.")

**Interpretación.** Debajo de la celda aparece el texto impreso y, a la izquierda, un número entre corchetes, por ejemplo `[1]`. Ese número confirma que la celda se ejecutó y en qué orden.

### Atajos de teclado esenciales

JupyterLab tiene dos modos. En **modo edición** (borde de la celda resaltado y cursor visible) se escribe dentro de la celda. En **modo comando** (se activa con `Esc`) se manipulan las celdas completas.

| Acción | Atajo | Modo |
|---|---|---|
| Ejecutar la celda y pasar a la siguiente | `Shift + Enter` | ambos |
| Ejecutar la celda y permanecer en ella | `Ctrl + Enter` | ambos |
| Ejecutar y crear una celda nueva debajo | `Alt + Enter` | ambos |
| Entrar a modo edición | `Enter` | comando |
| Salir a modo comando | `Esc` | edición |
| Insertar celda arriba / abajo | `A` / `B` | comando |
| Eliminar la celda | `D`, `D` (dos veces) | comando |
| Deshacer la eliminación de una celda | `Z` | comando |
| Convertir en celda Markdown / de código | `M` / `Y` | comando |
| Autocompletar nombres | `Tab` | edición |
| Ver la documentación de una función | `Shift + Tab` | edición |
| Guardar el cuaderno | `Ctrl + S` | ambos |

## 3. El estado del kernel y el orden de ejecución

El kernel conserva en memoria todas las variables creadas. Una variable definida en una celda está disponible en todas las celdas que se ejecuten después, sin importar su posición en la página. Las dos celdas siguientes lo demuestran.

In [ ]:
registros_procesados = 1_000_000     # se crea una variable en memoria
print("Variable creada")

In [ ]:
# Esta celda usa la variable creada en la celda anterior
print(f"Se procesaron {registros_procesados:,} registros")

**Interpretación.** La segunda celda funciona porque la primera se ejecutó antes. Si se reinicia el kernel y se ejecuta solo la segunda, aparece el error `NameError: name 'registros_procesados' is not defined`.

Por esta razón los cuadernos del taller deben ejecutarse **en orden**. Cuando el estado se vuelve confuso (se ejecutaron celdas salteadas o se modificó código anterior), la solución recomendada es:

- *Kernel > Restart Kernel and Run All Cells*: reinicia la memoria y ejecuta todo desde el principio.
- *Kernel > Restart Kernel*: reinicia sin ejecutar; luego se ejecutan las celdas manualmente.
- *Kernel > Interrupt Kernel* (botón ■): detiene una celda que tarda demasiado.

Al reiniciar el kernel también se cierra la sesión de Spark; la celda de preparación de cada cuaderno la vuelve a crear.

## 4. Lectura de mensajes de error

Un error no daña el cuaderno: solo detiene la celda donde ocurrió. El mensaje (*traceback*) se lee **de abajo hacia arriba**: la última línea indica el tipo de error y su causa, y las líneas anteriores indican dónde se produjo.

La celda siguiente provoca un error de forma controlada para mostrar su aspecto sin detener la ejecución del cuaderno.

In [ ]:
import traceback
try:
    total = "20" + 5      # error intencional: se suma texto con un número
except TypeError:
    traceback.print_exc(limit=1)

**Interpretación.** La última línea, `TypeError: can only concatenate str (not "int") to str`, explica la causa: no es posible sumar un texto y un número. En los cuadernos de Spark, los errores más frecuentes y su solución son:

| Mensaje | Causa habitual | Solución |
|---|---|---|
| `NameError: name 'df' is not defined` | No se ejecutó la celda de carga | Ejecutar las celdas anteriores en orden |
| `AnalysisException: [UNRESOLVED_COLUMN]` | Nombre de columna inexistente | Revisar `df.columns`; los nombres están normalizados (minúsculas, sin tildes) |
| `FileNotFoundError` | Ruta de archivo incorrecta | Revisar la ruta en el cuaderno 01 |
| `Py4JJavaError` | Error dentro de la JVM de Spark | Leer la línea `Caused by:` del mensaje |
| `Java gateway process exited` | Java no compatible o ausente | Ejecutar `python scripts/configurar_java.py` |

## 5. Celdas Markdown

Las celdas Markdown permiten documentar el análisis. Para editar una, se hace doble clic sobre ella; para mostrarla con formato, se ejecuta con `Shift + Enter`.

| Escritura | Resultado |
|---|---|
| `# Título`, `## Subtítulo` | Encabezados |
| `**negrita**`, `*cursiva*` | Énfasis |
| `- elemento` | Lista con viñetas |
| `` `código` `` | Código en línea |
| `[texto](https://...)` | Enlace |

La celda siguiente está destinada a practicar: se recomienda hacer doble clic y escribir una descripción breve del dataset que se analizará en el taller.

*Doble clic aquí para editar.* Dataset que se analizará: ...

## 6. Estructura de los cuadernos del taller

| Cuaderno | Contenido | Usa el dataset propio |
|---|---|---|
| `00_guia_uso_notebooks` | Uso de Jupyter (este cuaderno) | No |
| `01_configuracion_dataset` | Selección, carga y validación del dataset | Sí (se configura aquí) |
| `02_fundamentos_spark_rdd` | SparkSession, RDD, evaluación perezosa, particiones | Sí |
| `03_exploracion_dataframes` | Análisis exploratorio con DataFrames | Sí |
| `04_limpieza_etl_parquet` | Limpieza, transformación y escritura en Parquet | Sí (genera la versión limpia) |
| `05_spark_sql` | Consultas SQL, funciones de ventana | Sí |
| `06_mllib_modelos` | Regresión o clasificación y agrupamiento | Sí |
| `07_proyecto_final` | Plantilla del informe del estudiante | Sí |

El flujo de datos entre cuadernos es el siguiente:

```
archivo original ──► 01 configuración ──► data/dataset_config.json
                                               │
            ┌──────────────┬───────────────────┼──────────────────────┐
            ▼              ▼                   ▼                      ▼
      02 fundamentos  03 exploración   04 limpieza ──► data/procesado/<nombre>_limpio (Parquet)
                                                               │
                                                  ┌────────────┴───────────┐
                                                  ▼                        ▼
                                             05 Spark SQL            06 MLlib ──► 07 proyecto
```

En cada cuaderno, las celdas marcadas con `# >>> CELDA EDITABLE` son los puntos donde el estudiante adapta el análisis. Todo el taller funciona primero con el dataset de ejemplo; después, basta con configurar otro archivo en el cuaderno 01 y volver a ejecutar los demás.

## 7. Carga de un dataset propio

Existen varias formas de ubicar el archivo en la carpeta `data/mis_datos/` del taller:

| Entorno | Procedimiento |
|---|---|
| JupyterLab local | En el explorador de archivos, abrir `data/mis_datos/` y usar el botón *Upload* (flecha hacia arriba), o arrastrar el archivo desde el sistema operativo. |
| Explorador del sistema | Copiar el archivo en la carpeta `taller-apache-spark/data/mis_datos/`. |
| Docker | Copiar el archivo en `data/mis_datos/` en el equipo: la carpeta está montada dentro de los contenedores. |
| Google Colab | Ejecutar la celda de carga de Colab que aparece más abajo, o montar Google Drive. |

Formatos admitidos: CSV, TSV, TXT delimitado, JSON, JSON Lines, Parquet, ORC y Excel. La guía `docs/09-carga-de-datasets.md` explica cómo obtener datos abiertos, por ejemplo del Portal de Datos Abiertos de Colombia (Ministerio de Tecnologías de la Información y las Comunicaciones, s.f.).

La celda siguiente lista los archivos de datos disponibles. Después de cargar un archivo, se vuelve a ejecutar para confirmar que aparece.

La primera parte de la celda siguiente ubica la carpeta del taller. Esta misma preparación aparece al inicio de todos los cuadernos y evita los errores de ruta cuando el cuaderno se abre desde otra ubicación, como ocurre en Google Colab (carpeta de trabajo `/content`). En Colab debe reemplazarse `<usuario>` en `REPO_URL` por el usuario de GitHub donde está publicado el taller.

In [ ]:
import glob, os, subprocess, sys
from pathlib import Path

# >>> CELDA EDITABLE (solo Google Colab): dirección del repositorio del taller en GitHub
REPO_URL = "https://github.com/<usuario>/taller-apache-spark.git"

def en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def ubicar_taller():
    """Devuelve la carpeta raíz del taller: la que contiene scripts/taller_spark.py."""
    actual = Path.cwd().resolve()
    candidatos = [actual, *actual.parents]
    for patron in ("/content/*/scripts/taller_spark.py",                  # clonado en Colab
                   "/content/drive/MyDrive/*/scripts/taller_spark.py",    # copiado en Google Drive
                   "/content/drive/MyDrive/*/*/scripts/taller_spark.py"):
        candidatos += [Path(p).parent.parent for p in sorted(glob.glob(patron))]
    for base in candidatos:
        if (base / "scripts" / "taller_spark.py").exists():
            return base
    if en_colab():
        if "<usuario>" in REPO_URL:
            raise RuntimeError("En Google Colab debe escribirse en REPO_URL la dirección del repositorio "
                               "del taller (reemplazar <usuario>) y volver a ejecutar esta celda.")
        destino = Path("/content") / Path(REPO_URL.rstrip("/")).stem        # nombre del repositorio
        subprocess.run(["git", "clone", "-q", REPO_URL, str(destino)], check=True)
        return destino
    raise FileNotFoundError("No se encontró la carpeta del taller (scripts/taller_spark.py). "
                            "Abrir el cuaderno desde la carpeta notebooks/ del repositorio.")

RAIZ_TALLER = ubicar_taller()
os.chdir(RAIZ_TALLER / "notebooks")              # rutas iguales en cualquier entorno (local, Docker, Colab)
sys.path.insert(0, str(RAIZ_TALLER / "scripts"))

if en_colab():                                    # en Colab se instalan las dependencias que falten
    import importlib.util
    faltan = [m for m in ("openpyxl",) if importlib.util.find_spec(m) is None]
    try:
        import pyspark
        if int(pyspark.__version__.split(".")[0]) < 4:
            faltan.append("pyspark==4.3.0")
    except ImportError:
        faltan.append("pyspark==4.3.0")
    if faltan:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Carpeta del taller:", RAIZ_TALLER)

from taller_spark import listar_datasets, tamano_mb

for ruta in listar_datasets():
    print(f"{ruta:<55} {tamano_mb(ruta):8.2f} MB")

**Solo en Google Colab.** La celda siguiente está comentada para que no se ejecute en otros entornos. En Colab se eliminan los símbolos `#` y se ejecuta: aparecerá un botón para seleccionar el archivo.

In [ ]:
# from google.colab import files
# import shutil
# subidos = files.upload()                       # abre el selector de archivos
# for nombre in subidos:
#     destino = RAIZ_TALLER / "data" / "mis_datos" / nombre
#     shutil.move(nombre, destino)
#     print("Archivo guardado en", destino)

## 8. Comprobación de Spark desde el cuaderno

La última prueba consiste en crear una sesión de Spark y ejecutar un cálculo distribuido sencillo. Si aparece la versión de Spark y el resultado `500000500000`, el entorno está listo.

In [ ]:
from taller_spark import iniciar_spark

spark = iniciar_spark("00_Guia")
spark.range(1, 1_000_001).selectExpr("sum(id) AS suma").show()

**Interpretación.** `iniciar_spark()` imprime la versión de Spark, el modo de ejecución (`local[*]` usa todos los núcleos del equipo) y la dirección de la Spark UI. Al abrir esa dirección en el navegador, la pestaña *Jobs* muestra el trabajo que acaba de calcular la suma.

## 9. Buenas prácticas al trabajar con los cuadernos

1. **Guardar con frecuencia** (`Ctrl + S`). JupyterLab también crea puntos de control automáticos (*File > Revert Notebook to Checkpoint*).
2. **No modificar los cuadernos originales** para la entrega: se recomienda duplicarlos (*clic derecho > Duplicate*) o trabajar en el cuaderno `07_proyecto_final`.
3. **Limpiar las salidas antes de subir a GitHub** (*Edit > Clear Outputs of All Cells*), para evitar archivos pesados y conflictos.
4. **No usar `collect()` ni `toPandas()` sobre datos grandes**: traen todos los registros a la memoria del equipo. Para mirar datos se usan `show()`, `limit()` o `take()`.
5. **Cerrar la sesión** al terminar con `spark.stop()`, para liberar memoria.

## Ejercicios propuestos

1. Crear una celda nueva debajo de esta (modo comando, tecla `B`), escribir `2 ** 10` y ejecutarla.
2. Convertir esa celda en Markdown (tecla `M`) y escribir un título con `##`.
3. Reiniciar el kernel, ejecutar solo la celda de la sección 3 que imprime `registros_procesados` y explicar el error que aparece.
4. Cargar un archivo propio en `data/mis_datos/` y comprobar que aparece en la lista de la sección 7.

In [ ]:
spark.stop()   # libera los recursos de Spark al finalizar

## Referencias

Kluyver, T., Ragan-Kelley, B., Pérez, F., Granger, B., Bussonnier, M., Frederic, J., Kelley, K., Hamrick, J., Grout, J., Corlay, S., Ivanov, P., Avila, D., Abdalla, S., Willing, C., & Jupyter Development Team. (2016). Jupyter Notebooks: A publishing format for reproducible computational workflows. En F. Loizides & B. Schmidt (Eds.), *Positioning and power in academic publishing: Players, agents and agendas* (pp. 87–90). IOS Press. https://doi.org/10.3233/978-1-61499-649-1-87

Ministerio de Tecnologías de la Información y las Comunicaciones. (s.f.). *Portal de datos abiertos de Colombia* [Sitio web]. Recuperado el 9 de noviembre de 2026, de https://www.datos.gov.co/

Project Jupyter. (s.f.). *JupyterLab documentation* [Documentación técnica]. Recuperado el 9 de noviembre de 2026, de https://jupyterlab.readthedocs.io/